# Resolving association-test inputs from a `DonorData`

`cellink.at.GWAS`, `cellink.at.Skat` and `cellink.at.StructLMM` take a `DonorData` or
an `AnnData` and nothing else. Every input is a formula string resolved against that
object, and the variants come from the object itself, so a region is chosen by
subsetting rather than by passing a matrix. Two function families cross the levels:

- `crepeat(x)` broadcasts a **donor-level** value down to every cell of that donor.
- `dmean(x)`, `dfirst(x)`, `dmax(x)`, `dmedian(x)` aggregate a **cell-level** variable
  up to one value per donor.

The genotypes, donors and cell types below are real OneK1K; the expression is
simulated, so each test can be checked against a known ground truth.

## Load

`get_onek1k()` downloads ~17 GB and needs `PLINK` and `vcf2zarr` on `PATH`, so this
notebook is committed with its outputs (`nb_execution_mode = "off"` in `docs/conf.py`).

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from liftover import get_lifter

from cellink import DonorData
from cellink.at import GWAS, Skat, StructLMM, get_model_matrix
from cellink.resources import get_onek1k

rng = np.random.default_rng(0)
dd = get_onek1k()

[2026-09-24 14:11:16,700] WARNING:cellink.resources._datasets_utils: CELLINK_LIFTOVER_CACHE is not set; using default liftover cache at /Users/antonio.nappi/.liftover.
[2026-09-24 14:11:17,992] INFO:root: /Users/antonio.nappi/cellink_data/onek1k/onek1k_cellxgene.h5ad already exists
[2026-09-24 14:11:17,993] INFO:root: Veryifying checksum
[2026-09-24 14:11:20,904] INFO:root: /Users/antonio.nappi/cellink_data/onek1k/OneK1K.noGP.vcf.gz already exists
[2026-09-24 14:11:20,905] INFO:root: Veryifying checksum
[2026-09-24 14:11:28,040] INFO:root: /Users/antonio.nappi/cellink_data/onek1k/OneK1K.noGP.vcf.gz.csi already exists
[2026-09-24 14:11:28,040] INFO:root: Veryifying checksum
[2026-09-24 14:11:28,050] INFO:root: /Users/antonio.nappi/cellink_data/onek1k/gene_counts_Ensembl_105_phenotype_metadata.tsv.gz already exists
[2026-09-24 14:11:28,050] INFO:root: Veryifying checksum


/Users/antonio.nappi/miniforge3/envs/cellink/lib/python3.11/site-packages/sgkit/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution  # type: ignore[import]
/Users/antonio.nappi/miniforge3/envs/cellink/lib/python3.11/site-packages/sgkit/io/dataset.py:117: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of fall

## The variant

`rs1574036` is the memory-B lead eSNP for *EAF2* in OneK1K (Yazar et al., *Science*
**376**, eabf3041, 2022): a gene expressed across all cell types but genetically
controlled in only two of the study's fourteen -- immature/naive B and memory B. That
"expressed everywhere, regulated in one lineage" shape is exactly the pattern
simulated below, which is why this variant and this window. Its real dosages are used;
the expression is invented, so nothing here reproduces the published result.

### Mind the genome build

The OneK1K VCF is **GRCh38** while the published eSNP tables are **GRCh37**, so a
published position finds nothing until it is lifted. Ignore the `pos_hg19` column
`get_onek1k()` adds -- it lifts coordinates that are already hg38 and lands ~281 kb
off. `variant_id` holds positions rather than rsIDs, so there is no rsID lookup.

In [2]:
RS1574036_HG19 = 121_483_254  # rs1574036, as published
snp_pos = get_lifter("hg19", "hg38", one_based=True)["3"][RS1574036_HG19][0][1]

hits = dd.G.var[(dd.G.var["chrom"].astype(str) == "3") & (dd.G.var["pos"] == snp_pos)]
assert len(hits) == 1, f"expected one variant at 3:{snp_pos} (GRCh38), found {len(hits)}"

# the raw variant name starts with a digit, so copy the dosage to a formula-safe column
dd.G.obs["rs1574036"] = ad.utils.asarray(dd.G[:, hits.index[0]].X).ravel().astype(float)

# genotype PCs arrive with stringified integer column names; rename them so the `@`
# family operator can expand `@gPC[1:5]`
dd.G.obsm["gPCs"].columns = [f"gPC_{i}" for i in range(1, dd.G.obsm["gPCs"].shape[1] + 1)]

## Cell types

The expression half is Azimuth-annotated and carries none of the study's 14 labels, so
they are rebuilt here. `CD4_SOX4` and `CD8_S100B` are marker-defined and cannot be
recovered; the other twelve can. Check `dd.C.obs[label_col].value_counts()` first --
spellings differ between annotations.

In [3]:
label_col = "predicted.celltype.l2" if "predicted.celltype.l2" in dd.C.obs else "cell_type"
ONEK1K_MAP = {
    "CD4_NC": ["CD4 Naive", "CD4 TCM"],
    "CD4_ET": ["CD4 TEM", "CD4 CTL"],
    "CD8_NC": ["CD8 Naive", "CD8 TCM"],
    "CD8_ET": ["CD8 TEM"],
    "NK": ["NK"],
    "NK_R": ["NK_CD56bright"],
    "B_IN": ["B naive", "B intermediate"],
    "B_Mem": ["B memory"],
    "Plasma": ["Plasmablast"],
    "Mono_C": ["CD14 Mono"],
    "Mono_NC": ["CD16 Mono"],
    "DC": ["cDC1", "cDC2", "pDC", "ASDC"],
}
to_onek1k = {a: k for k, labels in ONEK1K_MAP.items() for a in labels}

dd.C.obs["cell_state"] = pd.Categorical(
    dd.C.obs[label_col].astype(str).map(to_onek1k).fillna("other"),
    categories=[*ONEK1K_MAP, "other"],
)
dd.C.obs["cell_state"].value_counts()

cell_state
CD4_NC     548012
NK         163202
CD8_ET     161051
B_IN        95591
CD8_NC      68947
other       63408
CD4_ET      49254
Mono_C      36130
B_Mem       30234
Mono_NC     15743
NK_R         7006
DC           6648
Plasma       3754
Name: count, dtype: int64

## Simulate a phenotype

Three genetic components, one per test:

1. a **burden** over the *cis* window, the same slope in every cell type -- `GWAS`
2. a **random genetic effect**, an independent draw per variant so signs cancel in a
   burden but still contribute variance across the set -- `Skat`
3. a **genotype x cell-type** effect, drawn only in the B states -- `StructLMM`

Each is a fraction of the cell-level variance; noise takes the remainder.

In [ ]:
VE_BURDEN, VE_RANDOM, VE_INTERACTION = 0.10, 0.07, 0.03
EFFECT_SIGN = -1.0
B_STATES = ("B_IN", "B_Mem")
WINDOW_BP, N_SET = 25_000, 100

window = dd.G.var[
    (dd.G.var["chrom"].astype(str) == "3") & (dd.G.var["pos"].between(snp_pos - WINDOW_BP, snp_pos + WINDOW_BP))
].copy()
if len(window) > N_SET:  # a diffuse effect over thousands of variants is undetectable
    window = window.loc[(window["pos"] - snp_pos).abs().nsmallest(N_SET).index].sort_values("pos")

X_raw = ad.utils.asarray(dd.G[:, window.index].X).astype(float)
alt_freq = 0.5 * X_raw.mean(axis=0)  # `X` counts a1, so fold it to get the MINOR allele
informative = np.minimum(alt_freq, 1.0 - alt_freq) > 0
X_raw, variant_names = X_raw[:, informative], window.index[informative]


def unit(x):
    """Centre and scale, so a sqrt(VE) coefficient is exactly that fraction."""
    return (x - x.mean()) / x.std()


burden = unit(X_raw.sum(axis=1))  # common variants, so weight 1 each
dd.G.obs["burden"] = burden
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
R = unit(X_std @ rng.normal(0, 1 / np.sqrt(X_raw.shape[1]), X_raw.shape[1]))

`gamma` is drawn over every state present, `"other"` included with zero: a state
missing from the index would make the `map()` below return `NaN`, and those cells
would silently have no phenotype. Noise takes the remainder of the **realised** slope,
since the burden contributes `(beta + gamma)^2` and the cross term does not cancel.

In [14]:
present = [s for s in dd.C.obs["cell_state"].cat.categories if (dd.C.obs["cell_state"] == s).any()]
cell_types = [s for s in present if s != "other"]

gamma = pd.Series([rng.normal(0, np.sqrt(VE_INTERACTION)) if s in B_STATES else 0.0 for s in present], index=present)
slope = EFFECT_SIGN * np.sqrt(VE_BURDEN) + gamma
ve_noise = 1.0 - slope**2 - VE_RANDOM
assert (ve_noise > 0).all(), "realised slope leaves no room for noise"

donor_ids = dd.C.obs[dd.donor_id].astype(str)  # a Categorical would poison the arithmetic
states = dd.C.obs["cell_state"].astype(str)
per_cell = pd.DataFrame({"burden": burden, "R": R}, index=dd.G.obs_names.astype(str)).reindex(donor_ids)

dd.C.obs["sim_expr"] = (
    states.map(slope).to_numpy() * per_cell["burden"].to_numpy()
    + np.sqrt(VE_RANDOM) * per_cell["R"].to_numpy()
    + np.sqrt(states.map(ve_noise).to_numpy()) * rng.normal(0, 1, dd.C.n_obs)
)
assert dd.C.obs["sim_expr"].notna().all(), "some cells have no simulated phenotype"

## The resolver on its own

`get_model_matrix(data, formula, target_level=...)` is what every model's `data=` path
calls. `sex` and `age` are stored per cell even though they are donor attributes, so a
donor-level model has to collapse them; `@gPC[1:5]` expands to the first five PCs.

In [15]:
get_model_matrix(dd, "dfirst(sex) + dfirst(age) + @gPC[1:5]", target_level="donor").head()

,Intercept,dfirst(sex),dfirst(age),gPC_1,gPC_2,gPC_3,gPC_4,gPC_5
donor_id,,,,,,,,
OneK1K_1,1.0,0,65,0.000550,-0.006975,-0.003147,0.002031,0.004651
OneK1K_10,1.0,0,78,0.001358,-0.008452,0.000481,-0.006151,0.003344
OneK1K_1000,1.0,0,62,0.001527,-0.002003,-0.008652,-0.002962,-0.001131
OneK1K_1001,1.0,0,73,-0.001149,0.001701,0.012521,-0.002202,0.011350
OneK1K_1002,1.0,0,57,-0.004588,0.002197,-0.004269,-0.004669,0.008571


The other direction: the donor genotype broadcast to every cell and crossed with the
cell state. This is the formula the numpy interface could not express without tiling
the genotype by hand and keeping it in sync with every cell filter.

In [16]:
get_model_matrix(dd, "crepeat(rs1574036) * cell_state", target_level="cell").columns.tolist()

['Intercept',
 'crepeat(rs1574036)',
 'cell_state[T.CD4_ET]',
 'cell_state[T.CD8_NC]',
 'cell_state[T.CD8_ET]',
 'cell_state[T.NK]',
 'cell_state[T.NK_R]',
 'cell_state[T.B_IN]',
 'cell_state[T.B_Mem]',
 'cell_state[T.Plasma]',
 'cell_state[T.Mono_C]',
 'cell_state[T.Mono_NC]',
 'cell_state[T.DC]',
 'cell_state[T.other]',
 'crepeat(rs1574036):cell_state[T.CD4_ET]',
 'crepeat(rs1574036):cell_state[T.CD8_NC]',
 'crepeat(rs1574036):cell_state[T.CD8_ET]',
 'crepeat(rs1574036):cell_state[T.NK]',
 'crepeat(rs1574036):cell_state[T.NK_R]',
 'crepeat(rs1574036):cell_state[T.B_IN]',
 'crepeat(rs1574036):cell_state[T.B_Mem]',
 'crepeat(rs1574036):cell_state[T.Plasma]',
 'crepeat(rs1574036):cell_state[T.Mono_C]',
 'crepeat(rs1574036):cell_state[T.Mono_NC]',
 'crepeat(rs1574036):cell_state[T.DC]',
 'crepeat(rs1574036):cell_state[T.other]']

## `GWAS`: one test per cell type

`dmean()` builds the pseudobulk phenotype inside the formula, over whatever cells the
object holds -- so subsetting to a cell type is what makes it a per-cell-type mean. A
derived vector such as the burden is not in `.X`, so it travels as its own `AnnData`.

In [17]:
MIN_CELLS, MIN_DONORS = 10, 100
COVARIATES = "dfirst(sex) + dfirst(age) + @gPC[1:5]"
counts = dd.C.obs.groupby([dd.donor_id, "cell_state"], observed=True).size().unstack(fill_value=0)

results, skipped = {}, {}
for state in cell_types:
    donors = counts.index[counts[state] >= MIN_CELLS]
    if len(donors) < MIN_DONORS:  # Plasma and DC lose most donors to the floor
        skipped[state] = len(donors)
        continue
    dd_s = dd[:, :, dd.C.obs["cell_state"].eq(state).to_numpy(), :].copy()
    dd_s = dd_s[donors, :, :, :].copy()

    tested = ad.AnnData(X=dd_s.G.obs[["burden", "rs1574036"]].to_numpy(dtype=float), obs=dd_s.G.obs)
    gwas = GWAS(Y="dmean(sim_expr)", F=COVARIATES, data=dd_s, target_level="donor")
    gwas.test_association(tested)

    pv = np.ravel(gwas.getPv())
    results[state] = {"n_donors": dd_s.G.n_obs, "p_burden": pv[0], "p_lead_snp": pv[1]}

print(f"tested {len(results)} cell types; skipped {skipped}")
pd.DataFrame(results).T

tested 11 cell types; skipped {'Plasma': 85}


,n_donors,p_burden,p_lead_snp
CD4_NC,981.0,0.000000e+00,4.946517e-261
CD4_ET,957.0,3.898355e-271,6.595798e-221
CD8_NC,938.0,7.864371e-272,5.570627e-216
CD8_ET,977.0,1.440106e-307,3.552569e-244
NK,979.0,0.000000e+00,2.730239e-248
NK_R,254.0,4.433593e-55,8.571008e-41
B_IN,968.0,2.990277e-300,1.559727e-236
B_Mem,870.0,2.414171e-244,1.066528e-199
Mono_C,660.0,1.906616e-176,9.363728e-142
Mono_NC,510.0,1.908644e-131,1.279872e-113


The burden is significant in **every** cell type, because its slope is non-zero
everywhere -- which says nothing about cell-type specificity. Only the interaction
below does. `p_lead_snp` is the honest unknown: rs1574036 is one variant inside a
100-variant burden, so how much signal it carries depends on real LD in the window.

## `StructLMM`: genotype x cell type

`E` is a formula like any other slot, resolved with no intercept since a constant
environment carries no interaction. The burden is donor-level while the phenotype is
cell-level, so it is handed over as a `DonorData` and StructLMM broadcasts it. The
result is **one p-value per variant column**, not one per cell: cells are the rows, and
the whole 12-state `E` is tested as a single variance component.

In [18]:
MAX_CELLS_PER_DONOR = 5  # a cell-level model over 12 types x 982 donors is too much

keep = dd.C.obs.loc[dd.C.obs["cell_state"].isin(cell_types)]
sampled = (
    keep.groupby([dd.donor_id, "cell_state"], observed=True)
    .apply(lambda df: df.sample(min(len(df), MAX_CELLS_PER_DONOR), random_state=0))
    .index.get_level_values(-1)
)
dd_i = dd[:, :, dd.C.obs_names.isin(sampled), :].copy()
# empty Categorical levels would give formulaic all-zero columns, i.e. a singular E
dd_i.C.obs["cell_state"] = dd_i.C.obs["cell_state"].cat.remove_unused_categories()


def burden_of(d):
    """Wrap the burden as the variant set of `d`; it is derived, so it is not in `.X`."""
    return DonorData(G=ad.AnnData(X=d.G.obs[["burden"]].to_numpy(dtype=float), obs=d.G.obs), C=d.C)


slmm = StructLMM(
    y="sim_expr",
    E="cell_state - 1",
    F="sex + age",  # already per cell, so no `crepeat` needed
    data=dd_i,
    target_level="cell",
)
p_all = slmm.interaction_test(burden_of(dd_i), exact=True)
p_all

/var/folders/r1/pqc5gkrn6rbbj5nksr0hgmh40000gn/T/ipykernel_92968/2605561366.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(min(len(df), MAX_CELLS_PER_DONOR), random_state=0))


[2026-09-24 14:24:00,369] INFO:root: Starting optimization ...


array([0.10211543])

That number says the slope varies *somewhere*, not where. Drop the two states `gamma`
was drawn in and it should go flat: the tested variant's own persistent effect is
already a fixed effect in the null, so only the variation across `E` is left. Print the
realised `gamma` too -- it is one draw per B state, and below `|gamma| ~ 0.05` there is
nothing to find at this sample size.

In [19]:
non_b = ~dd_i.C.obs["cell_state"].isin(B_STATES).to_numpy()
dd_nb = dd_i[:, :, non_b, :].copy()
dd_nb.C.obs["cell_state"] = dd_nb.C.obs["cell_state"].cat.remove_unused_categories()

slmm_nb = StructLMM(y="sim_expr", E="cell_state - 1", F="sex + age", data=dd_nb, target_level="cell")
p_nb = slmm_nb.interaction_test(burden_of(dd_nb), exact=True)

print(f"realised gamma: {gamma[list(B_STATES)].round(3).to_dict()}")
print(f"all 12 states: {p_all.ravel()[0]:.2e}    without B_IN/B_Mem: {p_nb.ravel()[0]:.3g}")

[2026-09-24 14:24:07,278] INFO:root: Starting optimization ...
realised gamma: {'B_IN': 0.0, 'B_Mem': 0.0}
all 12 states: 1.02e-01    without B_IN/B_Mem: 0.0806


Set `VE_INTERACTION = 0` and re-run: every marginal test above stays exactly as
significant while this interaction disappears. That is the difference between "this
gene has an eQTL" and "this gene has a *cell-type-specific* eQTL".

## `Skat`: the variant set

The set is whatever is in the object, so the window is chosen by subsetting. `a=1, b=1`
gives flat weights, matching the unweighted burden; the default `a=1, b=25` is a
rare-variant weighting and wrong for a window of common variants.

In [20]:
dd_b = dd[:, :, dd.C.obs["cell_state"].eq("B_Mem").to_numpy(), :].copy()
dd_b = dd_b[counts.index[counts["B_Mem"] >= MIN_CELLS], :, :, :].copy()
dd_cis = dd_b[:, variant_names, :, :].copy()

Skat(a=1, b=1, min_threshold=1).run_test(Y="dmean(sim_expr)", F="dfirst(sex) + dfirst(age)", data=dd_cis)

[2026-09-24 14:24:22,200] WARNING:cellink.at.skat: There are no variants with a MAC < 1.
[2026-09-24 14:24:22,205] INFO:root: Starting optimization ...


array([[5.69458669e-154]])

## Plain `AnnData`

`target_level` exists only because a `DonorData` has two levels. A plain `AnnData` has
one, so it is omitted -- and the aggregation functions are rejected there, which is why
the donor-level frame is collapsed with `dmean`/`dfirst` first and handed over ready-made.
The phenotype here is its own: an effect from rs1574036 and nothing else, so the number
below rests on the variant rather than on its LD with the rest of the burden.

In [21]:
VE_SNP = 0.05
g = unit(dd_b.G.obs["rs1574036"].to_numpy(dtype=float))

donor = dd_b.G.obs[["rs1574036"]].copy()
donor["expr"] = EFFECT_SIGN * np.sqrt(VE_SNP) * g + np.sqrt(1.0 - VE_SNP) * rng.normal(0, 1, len(g))
donor["age"] = get_model_matrix(dd_b, "dfirst(age) - 1", target_level="donor").to_numpy().ravel()

donor_adata = ad.AnnData(X=donor[["rs1574036"]].to_numpy(dtype=float), obs=donor[["expr", "age"]])
gwas_adata = GWAS(Y="expr", F="age", data=donor_adata)  # no `target_level`: there is only one
gwas_adata.test_association(donor_adata)  # the variants are this object's `.X`
gwas_adata.getPv()

array([[8.6821476e-15]])

## Things that bite on real data

- **Names must be unique.** A bare name is searched across `dd.G` and `dd.C`, in
  `obs`, `var`, `obsm`/`varm` and `X`; two matches raise `Key '...' is not unique`.
- **`crepeat` is for donor-level variables only.** `sex`/`age` live in `dd.C.obs`, so
  at cell level they are used directly -- `crepeat(sex)` raises there.
- **Subsetting cells changes the donor set.** `dd[:, :, cells, :]` re-intersects `G`
  against the donors still in `C`, so a donor with no cells left is removed, not left
  as `NaN`. Check `dd.G.n_obs` after a cell filter.
- **Genome build.** `dd.G.var["pos"]` is GRCh38; published tables are GRCh37.
- **Donors power these tests, not cells.** The phenotype is one number per donor no
  matter how many cells went into it.